In [94]:
from plot_utils import get_config_scores_dataframe, save_plot_to_memory, concatenate_and_save_pdfs, get_config_colours
import math
import plotnine as p9
from pathlib import Path
from IPython.display import Markdown as md
import os
import seaborn
from datetime import datetime
import re
import pingouin as pg
from glob import glob
from metrics.chexbert import PATHOLOGIES
from metrics.srr import SRRMetric
import warnings
import pandas as pd


In [95]:
fig_path = 'plots/cxrmate2/final.pdf'

num_plot_cols = 2
col_width = 7
geom_point_alpha = 0.6
geom_point_size = 2.5
geom_line_alpha = 0.6
text_size = 12
legend_columns = 2
plot_buffers = []
monitor_metric = 'RaTEScore (Findings)'

configs = [
    # {'path': '/scratch3/nic261/experiments/cxrmate2/final/000_sft', 'name': 'SFT', 'training_examples': 'NaN'},

    # {'path': '/scratch3/nic261/experiments/cxrmate2/final/medversa', 'name': 'MedVersa', 'training_examples': 'NaN'},

    {'path': '/scratch3/nic261/experiments/cxrmate2/final/emnli', 'name': 'EMNLI', 'training_examples': 152173, 'venue': "NAACL`21"},
    {'path': '/scratch3/nic261/experiments/cxrmate2/final/cxrmate', 'name': 'CXRMate', 'training_examples': 125395, 'venue': "IMU`24"},
    {'path': '/scratch3/nic261/experiments/cxrmate2/final/cxrmate_rrg24', 'name': 'CXRMate-RRG24', 'training_examples': 550395, 'venue': "BioNLP`24"},
    {'path': '/scratch3/nic261/experiments/cxrmate2/final/maira2', 'name': 'MAIRA-2', 'training_examples': 501825, 'venue': "`24"},
    {'path': '/scratch3/nic261/experiments/cxrmate2/final/medversa', 'name': 'MedVersa', 'training_examples': '-', 'venue': "`24"},
    {'path': '/scratch3/nic261/experiments/cxrmate2/final/libra', 'name': 'Libra', 'training_examples': 1213097, 'venue': "ACL`25"},
    {'path': '/scratch3/nic261/experiments/cxrmate2/final/cxrmate_ed', 'name': 'CXRMate-ED', 'training_examples': 76398, 'venue': "ACL`25"},
    {'path': '/scratch3/nic261/experiments/cxrmate2/final/medgemma', 'name': 'MedGemma', 'training_examples': 231483, 'venue': "`25"},
    {'path': '/scratch3/nic261/experiments/cxrmate2/final/001_grpo', 'name': 'CXRMate-2 (leg. A)', 'trai`ning_examples': 313503},
    {'path': '/scratch3/nic261/experiments/cxrmate2/final/002_grpo_rev_a', 'name': 'CXRMate-2', 'training_examples': 313503},
    # {'path': '/scratch3/nic261/experiments/cxrmate2/final/002_grpo_rev_a_one_epoch', 'name': 'CXRMate-2 (one epoch)', 'training_examples': 313503},
]   

Path(os.path.dirname(fig_path)).mkdir(parents=True, exist_ok=True)

df = get_config_scores_dataframe(configs)

md(f'[View PDF]({fig_path})')

[View PDF](plots/cxrmate2/final.pdf)

In [96]:
orig_num_plot_cols = num_plot_cols
orig_col_width = col_width
col_width = 4
num_plot_cols = 6

test_metrics = {
    'test_findings_ratescore_ratescore': 'RaTEScore (Findings)',
    'test_chexpert_plus_findings_ratescore_chexpert_plus_ratescore': 'RaTEScore (CheXpert Plus) (Findings)',
    'test_rexgradient_findings_ratescore_rexgradient_ratescore': 'RaTEScore (ReXgradient) (Findings)',
    'test_impression_ratescore_ratescore': 'RaTEScore (Impression)',
    'test_chexpert_plus_impression_ratescore_chexpert_plus_ratescore': 'RaTEScore (CheXpert Plus) (Impression)',
    'test_rexgradient_impression_ratescore_rexgradient_ratescore': 'RaTEScore (ReXgradient) (Impression)',

    'test_findings_radeval_bertscore_f1': 'RadEval BERTScore (Findings)',
    'test_chexpert_plus_findings_radeval_bertscore_chexpert_plus_f1': 'RadEval BERTScore(CheXpert Plus) (Findings)',
    'test_rexgradient_findings_radeval_bertscore_rexgradient_f1': 'RadEval BERTScore (ReXgradient) (Findings)',
    'test_impression_radeval_bertscore_f1': 'RadEval BERTScore (Impression)',
    'test_chexpert_plus_impression_radeval_bertscore_chexpert_plus_f1': 'RadEval BERTScore (CheXpert Plus) (Impression)',
    'test_rexgradient_impression_radeval_bertscore_rexgradient_f1': 'RadEval BERTScore (ReXgradient) (Impression)',

    'test_findings_srr_f1_macro': 'SRR F1 (Findings)',
    'test_chexpert_plus_findings_srr_chexpert_plus_f1_macro': 'SRR F1 (CheXpert Plus) (Findings)',
    'test_rexgradient_findings_srr_rexgradient_f1_macro': 'SRR F1 (ReXgradient) (Findings)',
    'test_impression_srr_f1_macro': 'SRR F1 (Impression)',
    'test_chexpert_plus_impression_srr_chexpert_plus_f1_macro': 'SRR F1 (CheXpert Plus) (Impression)',
    'test_rexgradient_impression_srr_rexgradient_f1_macro': 'SRR F1 (ReXgradient) (Impression)',

    'test_findings_radgraph-xl_rg_xl_rg_er': 'RadGraph-XL F1 (Findings)',
    'test_chexpert_plus_findings_radgraph-xl_chexpert_plus_rg_xl_rg_er': 'RadGraph-XL F1 (CheXpert Plus) (Findings)',
    'test_rexgradient_findings_radgraph-xl_rexgradient_rg_xl_rg_er': 'RadGraph-XL F1 (ReXgradient) (Findings)',
    'test_impression_radgraph-xl_rg_xl_rg_er': 'RadGraph-XL F1 (Impression)',
    'test_chexpert_plus_impression_radgraph-xl_chexpert_plus_rg_xl_rg_er': 'RadGraph-XL F1 (CheXpert Plus) (Impression)',
    'test_rexgradient_impression_radgraph-xl_rexgradient_rg_xl_rg_er': 'RadGraph-XL F1 (ReXgradient) (Impression)',

    'test_findings_chexbert_f1_macro': 'CheXbert F1 (Findings)',
    'test_chexpert_plus_findings_chexbert_chexpert_plus_f1_macro': 'CheXbert F1 (CheXpert Plus) (Findings)',
    'test_rexgradient_findings_chexbert_rexgradient_f1_macro': 'CheXbert F1 (ReXgradient) (Findings)',
    'test_impression_chexbert_f1_macro': 'CheXbert F1 (Impression)',
    'test_chexpert_plus_impression_chexbert_chexpert_plus_f1_macro': 'CheXbert F1 (CheXpert Plus) (Impression)',
    'test_rexgradient_impression_chexbert_rexgradient_f1_macro': 'CheXbert F1 (ReXgradient) (Impression)',

    'test_findings_cxrbert_similarity': 'CXR-BERT (Findings)',
    'test_chexpert_plus_findings_cxrbert_chexpert_plus_similarity': 'CXR-BERT (CheXpert Plus) (Findings)',
    'test_rexgradient_findings_cxrbert_rexgradient_similarity': 'CXR-BERT (ReXgradient) (Findings)',
    'test_impression_cxrbert_similarity': 'CXR-BERT (Impression)',
    'test_chexpert_plus_impression_cxrbert_chexpert_plus_similarity': 'CXR-BERT (CheXpert Plus) (Impression)',
    'test_rexgradient_impression_cxrbert_rexgradient_similarity': 'CXR-BERT (ReXgradient) (Impression)',

    'test_findings_bertscore_f1': 'BERTScore (Findings)',
    'test_chexpert_plus_findings_bertscore_chexpert_plus_f1': 'BERTScore (CheXpert Plus) (Findings)',
    'test_rexgradient_findings_bertscore_rexgradient_f1': 'BERTScore (ReXgradient) (Findings)',
    'test_impression_bertscore_f1': 'BERTScore (Impression)',
    'test_chexpert_plus_impression_bertscore_chexpert_plus_f1': 'BERTScore (CheXpert Plus) (Impression)',
    'test_rexgradient_impression_bertscore_rexgradient_f1': 'BERTScore (ReXgradient) (Impression)',

    'test_findings_rouge_l_f1': 'ROUGE-L (Findings)',
    'test_chexpert_plus_findings_rouge_l_chexpert_plus_f1': 'ROUGE-L (CheXpert Plus) (Findings)',
    'test_rexgradient_findings_rouge_l_rexgradient_f1': 'ROUGE-L (ReXgradient) (Findings)',
    'test_impression_rouge_l_f1': 'ROUGE-L (Impression)',
    'test_chexpert_plus_impression_rouge_l_chexpert_plus_f1': 'ROUGE-L (CheXpert Plus) (Impression)',
    'test_rexgradient_impression_rouge_l_rexgradient_f1': 'ROUGE-L (ReXgradient) (Impression)',

    'test_findings_bleu_bleu_4': 'B4 (Findings)',
    'test_chexpert_plus_findings_bleu_chexpert_plus_bleu_4': 'B4 (CheXpert Plus) (Findings)',
    'test_rexgradient_findings_bleu_rexgradient_bleu_4': 'B4 (ReXgradient) (Findings)',
    'test_impression_bleu_bleu_4': 'B4 (Impression)',
    'test_chexpert_plus_impression_bleu_chexpert_plus_bleu_4': 'B4 (CheXpert Plus) (Impression)',
    'test_rexgradient_impression_bleu_rexgradient_bleu_4': 'B4 (ReXgradient) (Impression)',

    'test_findings_arn_score': 'ARN (Findings)',
    'test_chexpert_plus_findings_arn_chexpert_plus_score': 'ARN (CheXpert Plus) (Findings)',
    'test_rexgradient_findings_arn_rexgradient_score': 'ARN (ReXgradient) (Findings)',
    'test_impression_arn_score': 'ARN (Impression)',
    'test_chexpert_plus_impression_arn_chexpert_plus_score': 'ARN (CheXpert Plus) (Impression)',
    'test_rexgradient_impression_arn_rexgradient_score': 'ARN (ReXgradient) (Impression)',

    'test_findings_green_green': 'GREEN (Findings)',
    'test_chexpert_plus_findings_green_chexpert_plus_green': 'GREEN (CheXpert Plus) (Findings)',
    'test_rexgradient_findings_green_rexgradient_green': 'GREEN (ReXgradient) (Findings)',
    'test_impression_green_green': 'GREEN (Impression)',
    'test_chexpert_plus_impression_green_chexpert_plus_green': 'GREEN (CheXpert Plus) (Impression)',
    'test_rexgradient_impression_green_rexgradient_green': 'GREEN (ReXgradient) (Impression)',
}

df_test = df[df['metric'].isin(list(test_metrics.keys()))].copy()
if len(df_test):
    df_test['metric'] = df_test['metric'].replace(test_metrics)
    df_test['metric'] = pd.Categorical(df_test['metric'], categories=list(dict.fromkeys(test_metrics.values())))
    df_test['config'] = df_test['config'].cat.remove_unused_categories()
    df_mean = df_test.drop(['trial'], axis=1).groupby(['config', 'metric'])['score'].mean().reset_index()
    df_std = df_test.drop(['trial'], axis=1).groupby(['config', 'metric'])['score'].std().reset_index()
    df_std['y_min'] = df_mean['score'].values - df_std['score'].fillna(0)
    df_std['y_max'] = df_mean['score'].values + df_std['score'].fillna(0)
    df_std['score'] = df_std['score'].fillna(0)
    num_plot_rows = math.ceil(df_test['metric'].nunique() / num_plot_cols)
    num_unique_configs = df_test['config'].nunique()
    plot = (
        p9.ggplot()
        + p9.geom_errorbar(df_std, p9.aes(x='config', ymin='y_min', ymax='y_max'), alpha=1.0, color='#666666')
        + p9.geom_point(df_mean, p9.aes(x='config', y='score', fill='config'), show_legend=False, shape='+', alpha=1.0, color='#666666')
        + p9.geom_point(df_test, p9.aes(x='config', y='score', fill='config'), show_legend=False, stroke=0, size=geom_point_size, alpha=geom_point_alpha)
        + p9.facet_wrap(['metric'], ncol=num_plot_cols, scales='free_x')
        + p9.theme_minimal()
        + p9.theme(
            figure_size=(num_plot_cols * col_width, 0.5 + (num_unique_configs) + num_plot_rows),
            text=p9.element_text(size=text_size),
        )
        + p9.coord_flip() + p9.xlab('') + p9.ylab('')
    )
    plot_buffers.append(save_plot_to_memory(plot))

col_width = orig_col_width
num_plot_cols = orig_num_plot_cols

/tmp/ipykernel_48412/2738749268.py:90: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
/tmp/ipykernel_48412/2738749268.py:91: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
/scratch3/nic261/environments/cxrmate2_virga_final/lib/python3.12/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_errorbar : Removed 235 rows containing missing values.
/scratch3/nic261/environments/cxrmate2_virga_final/lib/python3.12/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_point : Removed 235 rows containing missing values.


In [97]:
# No. examples per set:
df_set = df[~df['stage'].isin(['train_step', 'train_epoch', 'val', 'test'])][['config', 'trial', 'stage', 'score']]
score_counts = df_set.groupby(['config', 'stage'])['score'].nunique()
if not score_counts.le(1).all():
    warnings.warn(f"Inconsistent scores across trials: {score_counts[score_counts > 1]}")
df_set.head()
plot = (
    p9.ggplot(df_set, p9.aes(x='stage', y='score', fill='config'))
    + p9.geom_col(stat='identity', position='dodge')
    + p9.geom_text(p9.aes(label='score'), position=p9.position_dodge(width=0.9), size=8, va='bottom')
    + p9.xlab('Configuration') + p9.ylab('Epochs')
    + p9.theme(
        axis_text_x=p9.element_text(rotation=45, ha='right'), figure_size=(14, 6),
        text=p9.element_text(size=text_size),
    )
)
plot_buffers.append(save_plot_to_memory(plot))

/tmp/ipykernel_48412/3324047523.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
/tmp/ipykernel_48412/3324047523.py:5: UserWarning: Inconsistent scores across trials: config              stage         
CXRMate-2 (leg. A)  MIMIC-CXR test    2
Name: score, dtype: int64


In [98]:
train_metrics = {
    'seq_len': 'Sequence Length',
    'advantages_mean': 'Advantage',
    'train_loss': 'Loss',
    'scheduler_lr': 'Scheduler LR',
    'kl': 'KL Divergence',
    'reward_ratescore_mean': 'RaTEScore mean',
    'reward_ratescore_std': 'RaTEScore std',
    'reward_cxrbert_mean': 'CXR-BERT mean',
    'reward_cxrbert_std': 'CXR-BERT std',
    'reward_bertscore_mean': 'BERTScore mean',
    'reward_bertscore_std': 'BERTScore std',
    'reward_arn_mean': 'ARN mean',
    'reward_arn_std': 'ARN std',
    'prompt_len': 'Prompt Length',
    'gpu_allocated_memory_gb': 'GPU Allocated Memory (GB)',
    'gpu_reserved_memory_gb': 'GPU Reserved Memory (GB)'
}

df_train = df[df['metric'].isin(list(train_metrics.keys()))].copy()
if len(df_train):
    df_train['metric'] = df_train['metric'].replace(train_metrics)
    df_train['metric'] = pd.Categorical(df_train['metric'], categories=list(dict.fromkeys(train_metrics.values())))
    df_train['config'] = df_train['config'].cat.remove_unused_categories()
    num_plot_rows = math.ceil(df_train['metric'].nunique() / num_plot_cols)
    config_colours = get_config_colours(df_train['config'].unique().tolist(), df_train['config_trial'].unique().tolist())
    plot = (
        p9.ggplot()
        + p9.geom_line(df_train, p9.aes(x='step', y='score', color='config_trial'), alpha=geom_line_alpha, show_legend=False)
        + p9.geom_point(df_train, p9.aes(x='step', y='score', fill='config'), show_legend=True, stroke=0, size=geom_point_size, alpha=geom_point_alpha)
        + p9.facet_wrap(['metric'], ncol=num_plot_cols, scales='free')
        + p9.scale_color_manual(values=config_colours)
        + p9.scale_fill_manual(values=config_colours)
        + p9.guides(fill=p9.guide_legend(ncol=legend_columns))
        + p9.theme_minimal()
        + p9.theme(
            figure_size=(num_plot_cols * col_width, num_plot_rows * 5), 
            legend_title=p9.element_blank(),
            legend_position='top',
            text=p9.element_text(size=text_size)
        )
        + p9.xlab('') + p9.ylab('')
    )
    plot_buffers.append(save_plot_to_memory(plot))


In [99]:
val_metrics = {
    'val_findings_green_green': 'GREEN',
    'val_findings_ratescore_ratescore': 'RaTEScore',
    'val_findings_srr_f1_macro': 'SRR F1',
    'val_findings_chexbert_f1_macro': 'CheXbert F1',
    'val_findings_cxrbert_similarity': 'CXR-BERT',
    'val_findings_bertscore_f1': 'BERTScore',
    'val_findings_rouge_l_f1': 'ROUGE-L',
    'val_findings_bleu_bleu_4': 'B4',
    'val_findings_arn_score': 'ARN',
}

df_val = df[df['metric'].isin(list(val_metrics.keys()))].copy()
df_val['metric'] = df_val['metric'].replace(val_metrics)
df_val['metric'] = pd.Categorical(df_val['metric'], categories=list(dict.fromkeys(val_metrics.values())))
df_val['config'] = df_val['config'].cat.remove_unused_categories()
num_plot_rows = math.ceil(df_val['metric'].nunique() / num_plot_cols)
config_colours = get_config_colours(df_val['config'].unique().tolist(), df_val['config_trial'].unique().tolist())

for i in ['step', 'epoch']:

    # Validation:
    if len(df_val):
        plot = (
            p9.ggplot()
            + p9.geom_line(df_val, p9.aes(x=i, y='score', color='config_trial'), alpha=geom_line_alpha, show_legend=False)
            + p9.geom_point(df_val, p9.aes(x=i, y='score', fill='config'), show_legend=True, stroke=0, size=geom_point_size, alpha=geom_point_alpha)
            + p9.facet_wrap(['metric'], ncol=num_plot_cols, scales='free')
            + p9.scale_color_manual(values=config_colours)
            + p9.scale_fill_manual(values=config_colours)
            + p9.guides(fill=p9.guide_legend(ncol=legend_columns))
            + p9.theme_minimal()
            + p9.theme(
                figure_size=(num_plot_cols * col_width, num_plot_rows * 5), 
                legend_title=p9.element_blank(),
                legend_position='top',
                text=p9.element_text(size=text_size)
            )
            + p9.xlab('') + p9.ylab('')
        )
        plot_buffers.append(save_plot_to_memory(plot))

In [100]:
# Epochs:
df_epoch = df.groupby(['config', 'trial'])['epoch'].max().reset_index()
df_epoch['trial'] = pd.Categorical(df_epoch['trial'])
df_epoch.dropna(subset=['epoch'], inplace=True)
df_epoch['epoch'] = df_epoch['epoch'].astype(int)
plot = (
    p9.ggplot(df_epoch, p9.aes(x='config', y='epoch', fill='trial'))
    + p9.geom_col(stat='identity', position='dodge')
    + p9.geom_text(p9.aes(label='epoch'), position=p9.position_dodge(width=0.9), size=8, va='bottom')
    + p9.xlab('Configuration') + p9.ylab('Epochs')
    + p9.theme(
        axis_text_x=p9.element_text(rotation=45, ha='right'), figure_size=(num_plot_cols * col_width, 5),
        text=p9.element_text(size=text_size),
    )
)
plot_buffers.append(save_plot_to_memory(plot))


/tmp/ipykernel_48412/8309104.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


In [101]:
# Steps:
df_step = df.groupby(['config', 'trial'])['step'].max().reset_index()
df_step['trial'] = pd.Categorical(df_step['trial'])
df_step.dropna(subset=['step'], inplace=True)
df_step['step'] = df_step['step'].astype(int)
plot = (
    p9.ggplot(df_step, p9.aes(x='config', y='step', fill='trial'))
    + p9.geom_col(stat='identity', position='dodge')
    + p9.geom_text(p9.aes(label='step'), angle=45, position=p9.position_dodge(width=0.9), size=8, va='bottom', ha='left')
    + p9.xlab('Configuration') + p9.ylab('Steps')
    + p9.theme(
        axis_text_x=p9.element_text(rotation=45, ha='right'), 
        figure_size=(num_plot_cols * col_width, 5),
        text=p9.element_text(size=text_size),
    )
)
plot_buffers.append(save_plot_to_memory(plot))

/tmp/ipykernel_48412/3606103052.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


In [102]:
# SRR pathology bar plots:
for test_set, title in zip(['', '_chexpert_plus', '_rexgradient'], ['MIMIC-CXR', 'CheXpert Plus', 'RexGradient'], strict=True):
    srr_metrics = {f'test{test_set}_findings_srr{test_set}_f1_{i}': i.replace('_', ' ').capitalize() for i in list(SRRMetric.idx_2_label.values())}
    df_srr_labels = df[df['metric'].isin(list(srr_metrics.keys()))].copy()
    if len(df_srr_labels):      
        df_srr_labels['metric'] = df_srr_labels['metric'].replace(srr_metrics)
        df_srr_labels['metric'] = pd.Categorical(df_srr_labels['metric'], categories=list(srr_metrics.values()))
        df_srr_labels.tail()
        df_mean = df_srr_labels.drop(['trial'], axis=1).groupby(['config', 'metric'])['score'].mean().reset_index()
        position = p9.position_dodge(width = 0.75)
        plot = (
            p9.ggplot()
            + p9.geom_col(df_mean, p9.aes(x='metric', y='score', fill='config'), position=position, width=1.)
            + p9.scale_fill_manual(values=list(seaborn.color_palette(palette='colorblind', n_colors=df_mean['config'].nunique()).as_hex()))
            + p9.ylim(0.0, 1.0)
            + p9.ggtitle(title)
            + p9.ylab('SRR F1')
            + p9.xlab('')
            + p9.theme_tufte()
            + p9.theme(
                axis_text_x=p9.element_text(rotation=45, ha='right', margin={'t': -4.5}),
                legend_title=p9.element_blank(),
                axis_ticks_major_x=p9.element_blank(),
                figure_size=(num_plot_cols * col_width, 5),
                legend_position=(0.5, 0.875),
                axis_title=p9.element_text(size=14),
                axis_text=p9.element_text(size=12),
                legend_text=p9.element_text(size=12),
                plot_title=p9.element_text(size=14),
                strip_text=p9.element_text(size=14)
            )
        )
        plot_buffers.append(save_plot_to_memory(plot))

/tmp/ipykernel_48412/2439409802.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
/scratch3/nic261/environments/cxrmate2_virga_final/lib/python3.12/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_col : Removed 104 rows containing missing values.
/tmp/ipykernel_48412/2439409802.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
/scratch3/nic261/environments/cxrmate2_virga_final/lib/python3.12/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_col : Removed 371 rows containing missing values.
/tmp/ipykernel_48412/2439409802.py:9: FutureWarning: The default of observed=False is deprecated and w

In [103]:
# CheXbert pathology bar plots:
for test_set, title in zip(['', '_chexpert_plus', '_rexgradient'], ['MIMIC-CXR', 'CheXpert Plus', 'RexGradient'], strict=True):
    chexbert_metrics = {f'test{test_set}_findings_chexbert{test_set}_f1_{i}': i.replace('_', ' ').capitalize() for i in PATHOLOGIES}
    df_chexpert_labels = df[df['metric'].isin(list(chexbert_metrics.keys()))].copy()
    if len(df_chexpert_labels):      
        df_chexpert_labels['metric'] = df_chexpert_labels['metric'].replace(chexbert_metrics)
        df_chexpert_labels['metric'] = pd.Categorical(df_chexpert_labels['metric'], categories=list(chexbert_metrics.values()))
        df_chexpert_labels.tail()
        df_mean = df_chexpert_labels.drop(['trial'], axis=1).groupby(['config', 'metric'])['score'].mean().reset_index()
        position = p9.position_dodge(width = 0.75)
        plot = (
            p9.ggplot()
            + p9.geom_col(df_mean, p9.aes(x='metric', y='score', fill='config'), position=position, width=1.)
            + p9.scale_fill_manual(values=list(seaborn.color_palette(palette='colorblind', n_colors=df_mean['config'].nunique()).as_hex()))
            + p9.ylim(0.0, 1.0)
            + p9.ggtitle(title)
            + p9.ylab('CheXbert F1')
            + p9.xlab('')
            + p9.theme_tufte()
            + p9.theme(
                axis_text_x=p9.element_text(rotation=45, ha='right', margin={'t': -4.5}),
                legend_title=p9.element_blank(),
                axis_ticks_major_x=p9.element_blank(),
                figure_size=(num_plot_cols * col_width, 5),
                legend_position=(0.5, 0.875),
                axis_title=p9.element_text(size=14),
                axis_text=p9.element_text(size=12),
                legend_text=p9.element_text(size=12),
                plot_title=p9.element_text(size=14),
                strip_text=p9.element_text(size=14)
            )
        )
        plot_buffers.append(save_plot_to_memory(plot))

/tmp/ipykernel_48412/1448065019.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
/tmp/ipykernel_48412/1448065019.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
/scratch3/nic261/environments/cxrmate2_virga_final/lib/python3.12/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_col : Removed 43 rows containing missing values.
/tmp/ipykernel_48412/1448065019.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
/scr

In [104]:
concatenate_and_save_pdfs(plot_buffers, fig_path)
md(f'[View PDF]({fig_path})')

[View PDF](plots/cxrmate2/final.pdf)

In [105]:
best = df_test[df_test['metric'] == monitor_metric].copy()
idx = best.groupby(['config', 'trial'], observed=True)['score'].idxmax()
best = (
    best.loc[idx, ['config', 'trial', 'epoch', 'score']]
    .sort_values(['config', 'trial'])
    .reset_index(drop=True)
 )
idx = best.groupby(['config'], observed=True)['score'].idxmax()
best = best.loc[idx]

missing = [i for i in df_test['config'].cat.categories if i not in set(best['config'])]

df_missing = df_test[df_test['config'].isin(missing)].copy()

counts = df_missing.dropna(subset=['trial']).groupby('config', observed=True)['trial'].nunique().to_frame()
display(f'Configs without metric: {monitor_metric} and more than one trial: {counts[counts['trial'] > 1].index.tolist()}.')
missing = [i for i in missing if i in set(counts[counts['trial'] == 1].index.tolist())]

counts = df_missing.dropna(subset=['epoch']).groupby('config', observed=True)['epoch'].nunique().to_frame()
display(f'Configs without metric: {monitor_metric} and more than one epoch: {counts[counts['epoch'] > 1].index.tolist()}.')
missing = [i for i in missing if i in set(counts[counts['epoch'] == 1].index.tolist())]

missing = [
    {
        'config': i, 
        'trial': df_test[df_test['config'] == i]['trial'].unique().item(),
        'epoch': df_test[df_test['config'] == i]['epoch'].unique().item(),
    } 
    for i in missing
]
best = pd.concat([best, pd.DataFrame(missing)], ignore_index=True)
best


'Configs without metric: RaTEScore (Findings) and more than one trial: [].'

'Configs without metric: RaTEScore (Findings) and more than one epoch: [].'

,config,trial,epoch,score
0,EMNLI,0,-1,0.592443
1,CXRMate,0,-1,0.566622
2,CXRMate-RRG24,0,-1,0.580584
3,MAIRA-2,0,-1,0.592464
4,MedVersa,0,-1,0.605962
5,Libra,0,-1,0.537473
6,CXRMate-ED,0,-1,0.598919
7,MedGemma,0,-1,0.573275
8,CXRMate-2 (leg. A),7,-1,0.678161
9,CXRMate-2,6,-1,0.675725


In [106]:
"""
Resources for stats:
https://www.statsflowchart.co.uk/
https://statsandr.com/blog/files/overview-statistical-tests-statsandr.pdf
https://statsandr.com/blog/anova-in-r/#introduction
"""

section = 'findings'

test_metrics = {
    'RaTEScore MIMIC-CXR': {'dir_name': 'ratescore', 'column_name': 'ratescore', 'csv_name': f'test_{section}_*_scores_*'},
    'RaTEScore CheXpert Plus': {'dir_name': 'ratescore_chexpert_plus', 'column_name': 'ratescore', 'csv_name': f'test_chexpert_plus_{section}_*_scores_*'},
    'RaTEScore ReXgradient': {'dir_name': 'ratescore_rexgradient', 'column_name': 'ratescore', 'csv_name': f'test_rexgradient_{section}_*_scores_*'},

    'GREEN MIMIC-CXR': {'dir_name': 'green', 'column_name': 'green', 'csv_name': f'test_{section}_*_scores_*'},
    'GREEN CheXpert Plus': {'dir_name': 'green_chexpert_plus', 'column_name': 'green', 'csv_name': f'test_chexpert_plus_{section}_*_scores_*'},
    'GREEN ReXgradient': {'dir_name': 'green_rexgradient', 'column_name': 'green', 'csv_name': f'test_rexgradient_{section}_*_scores_*'},

    'RadEval BERTScore MIMIC-CXR': {'dir_name': 'radeval_bertscore', 'column_name': 'f1', 'csv_name': f'test_{section}_*_scores_*'},
    'RadEval BERTScore CheXpert Plus': {'dir_name': 'radeval_bertscore_chexpert_plus', 'column_name': 'f1', 'csv_name': f'test_chexpert_plus_{section}_*_scores_*'},
    'RadEval BERTScore ReXgradient': {'dir_name': 'radeval_bertscore_rexgradient', 'column_name': 'f1', 'csv_name': f'test_rexgradient_{section}_*_scores_*'},

    'SRR MIMIC-CXR': {'metric_type': 'classification', 'df_test_metric_name': 'SRR F1 (Findings)', 'df_count_name': 'test_findings_srr_num_study_ids'},
    'SRR CheXpert Plus': {'metric_type': 'classification', 'df_test_metric_name': 'SRR F1 (CheXpert Plus) (Findings)', 'df_count_name': 'test_chexpert_plus_findings_srr_chexpert_plus_num_study_ids'},
    'SRR ReXgradient': {'metric_type': 'classification', 'df_test_metric_name': 'SRR F1 (ReXgradient) (Findings)', 'df_count_name': 'test_rexgradient_findings_srr_rexgradient_num_study_ids'},

    'RadGraph-XL MIMIC-CXR': {'dir_name': 'radgraph-xl', 'column_name': 'rg_xl_rg_er', 'csv_name': f'test_{section}_*_scores_*'},
    'RadGraph-XL CheXpert Plus': {'dir_name': 'radgraph-xl_chexpert_plus', 'column_name': 'rg_xl_rg_er', 'csv_name': f'test_chexpert_plus_{section}_*_scores_*'},
    'RadGraph-XL ReXgradient': {'dir_name': 'radgraph-xl_rexgradient', 'column_name': 'rg_xl_rg_er', 'csv_name': f'test_rexgradient_{section}_*_scores_*'},

    'CheXbert MIMIC-CXR': {'metric_type': 'classification', 'df_test_metric_name': 'CheXbert F1 (Findings)', 'df_count_name': 'test_findings_chexbert_num_study_ids'},
    'CheXbert CheXpert Plus': {'metric_type': 'classification', 'df_test_metric_name': 'CheXbert F1 (CheXpert Plus) (Findings)', 'df_count_name': 'test_chexpert_plus_findings_chexbert_chexpert_plus_num_study_ids'},
    'CheXbert ReXgradient': {'metric_type': 'classification', 'df_test_metric_name': 'CheXbert F1 (ReXgradient) (Findings)', 'df_count_name': 'test_rexgradient_findings_chexbert_rexgradient_num_study_ids'},

    'CXR-BERT MIMIC-CXR': {'dir_name': 'cxrbert', 'column_name': 'similarity', 'csv_name': f'test_{section}_*_scores_*'},
    'CXR-BERT CheXpert Plus': {'dir_name': 'cxrbert_chexpert_plus', 'column_name': 'similarity', 'csv_name': f'test_chexpert_plus_{section}_*_scores_*'},
    'CXR-BERT ReXgradient': {'dir_name': 'cxrbert_rexgradient', 'column_name': 'similarity', 'csv_name': f'test_rexgradient_{section}_*_scores_*'},

    'BERTScore MIMIC-CXR': {'dir_name': 'bertscore', 'column_name': 'f1', 'csv_name': f'test_{section}_*_scores_*'},
    'BERTScore CheXpert Plus': {'dir_name': 'bertscore_chexpert_plus', 'column_name': 'f1', 'csv_name': f'test_chexpert_plus_{section}_*_scores_*'},
    'BERTScore ReXgradient': {'dir_name': 'bertscore_rexgradient', 'column_name': 'f1', 'csv_name': f'test_rexgradient_{section}_*_scores_*'},

    'BLEU MIMIC-CXR': {'dir_name': 'bleu', 'column_name': 'bleu_4', 'csv_name': f'test_{section}_*_scores_*'},
    'BLEU CheXpert Plus': {'dir_name': 'bleu_chexpert_plus', 'column_name': 'bleu_4', 'csv_name': f'test_chexpert_plus_{section}_*_scores_*'},
    'BLEU ReXgradient': {'dir_name': 'bleu_rexgradient', 'column_name': 'bleu_4', 'csv_name': f'test_rexgradient_{section}_*_scores_*'},

    'ROUGE-L MIMIC-CXR': {'dir_name': 'rouge_l', 'column_name': 'f1', 'csv_name': f'test_{section}_*_scores_*'},
    'ROUGE-L CheXpert Plus': {'dir_name': 'rouge_l_chexpert_plus', 'column_name': 'f1', 'csv_name': f'test_chexpert_plus_{section}_*_scores_*'},
    'ROUGE-L ReXgradient': {'dir_name': 'rouge_l_rexgradient', 'column_name': 'f1', 'csv_name': f'test_rexgradient_{section}_*_scores_*'},

    # 'ARN MIMIC-CXR': {'dir_name': 'arn', 'column_name': 'score', 'csv_name': f'test_{section}_*_scores_*'},
    # 'ARN CheXpert Plus': {'dir_name': 'arn_chexpert_plus', 'column_name': 'score', 'csv_name': f'test_chexpert_plus_{section}_*_scores_*'},
    # 'ARN ReXgradient': {'dir_name': 'arn_rexgradient', 'column_name': 'score', 'csv_name': f'test_rexgradient_{section}_*_scores_*'},
}

p_sig = 0.05

significant = {}
csv_paths = []
significant_effective = {}

def strict_winner(df_posthoc, models, alpha=0.05):
    pcol = 'p-tukey' if 'p-tukey' in df_posthoc.columns else 'pval'
    for m in models:
        rel = df_posthoc[(df_posthoc['A'] == m) | (df_posthoc['B'] == m)]
        if rel.empty or len(rel) < len(models) - 1:
            continue  # missing comparisons
        ok = []
        for _, r in rel.iterrows():
            if r['A'] == m:
                ok.append((r['diff'] > 0) and (r[pcol] < alpha))
            elif r['B'] == m:
                ok.append((r['diff'] < 0) and (r[pcol] < alpha))
        if all(ok):
            return m
    return None

for metric_name, metric_config in test_metrics.items():

    if 'metric_type' in metric_config and metric_config['metric_type'] == 'classification':
        accumulated = {}
        count = {}
        for _, row in best.iterrows():

            score = df_test[
                (df_test['config'] == row['config']) & \
                (df_test['metric'] == metric_config['df_test_metric_name']) & \
                (df_test['trial'] == row['trial']) & \
                (df_test['epoch'] == row['epoch'])
            ]['score']
            
            if len(score) != 1:
                continue

            count[row['config']] = df[
                (df['config'] == row['config']) & \
                (df['metric'] == metric_config['df_count_name']) & \
                (df['trial'] == row['trial']) & \
                (df['epoch'] == row['epoch'])
            ]['score'].item()

            if count[row['config']] <= 16:
                warnings.warn(f'Only {count[row["config"]]} samples for {metric_name} in {row["config"]} trial {row["trial"]} epoch {row["epoch"]}. Skipping.')
                continue


            accumulated[row['config']] = score.item()

        test_metrics[metric_name]['accumulated'] = accumulated
        test_metrics[metric_name]['count'] = count

    else:
        
        df_list = []
        for _, row in best.iterrows():


            if metric_name == 'RadEval BERTScore MIMIC-CXR' and row['config'] == 'CXRMate-2':
                pass


            path = [config['path'] for config in configs if config['name'] == row['config']]
            assert len(path) == 1
            path = path[0]
            csv_path_candidates = glob(f'{path}/trial_{row["trial"]}/metric_outputs/{metric_config["dir_name"]}/{metric_config["csv_name"]}.csv', recursive=True)

            # Keep only csv_paths that match the best epoch for this config/trial:
            csv_paths = [
                p for p in csv_path_candidates
                if re.search(r'epoch-(-?\d+)', p) and int(re.search(r'epoch-(-?\d+)', p).group(1)) ==  row['epoch']
            ]

            # Fallback to epoch=-1 if best epoch not found:
            if len(csv_paths) == 0:
                csv_paths = [
                    p for p in csv_path_candidates
                    if re.search(r'epoch-(-?\d+)', p) and int(re.search(r'epoch-(-?\d+)', p).group(1)) ==  -1
                ]
                
            if len(csv_paths) == 0:
                warnings.warn(f"No CSV found for best epoch {row['epoch']} in {path} trial {row['trial']}.")
                continue
            
            csv_path = max(
                csv_paths,
                key=lambda p: datetime.strptime(
                    re.search(r'(\d{2}-\d{2}-\d{4}_\d{2}-\d{2}-\d{2})', p).group(1),
                    "%d-%m-%Y_%H-%M-%S"
                )
            )
            
            df_trial = pd.read_csv(csv_path)

            if len(df_trial) <= 16:
                warnings.warn(f'{csv_path} has 16 or less rows ({len(df_trial)}).')
                continue

            df_trial['config'] = row['config']
            df_trial['trial'] = row['trial']
            df_trial['score'] = df_trial[metric_config['column_name']]

            df_trial = df_trial[['config', 'trial', 'study_id', 'score']]

            df_list.append(df_trial)        

        if df_list:
            df_metric = pd.concat(df_list, ignore_index=True, axis=0)
        else:
            continue

        print(f'{metric_config["dir_name"]}, {metric_config["column_name"]}:')

        n = list(set(df_metric['config'].value_counts().to_list()))

        print(f'No. samples: {n[-1]}.')

        if df_metric['config'].nunique() == 1:
            pass
        elif df_metric['config'].nunique() == 2:

            # Welch t-test (robust to unequal variances):
            levels = df_metric['config'].dropna().unique().tolist()
            x = df_metric.loc[df_metric['config'] == levels[0], 'score']
            y = df_metric.loc[df_metric['config'] == levels[1], 'score']

            t_res = pg.ttest(x, y, paired=False, alternative='two-sided', correction='auto')  # Welch by default with correction='auto'
            p = float(t_res['p-val'].iloc[0])
            test_metrics[metric_name]['ttest'] = t_res
            print(f"\tTwo-group Welch t-test, p<{p_sig}:", p < p_sig)

            winner = None
            if p < p_sig:
                winner = levels[0] if x.mean() > y.mean() else levels[1]

            test_metrics[metric_name]['winner'] = winner
            print(f"\tTwo-group winner: {winner if winner else 'None'}")

        else:

            # Homogeneity of variances (Levene)
            res = pg.homoscedasticity(data=df_metric, group='config', dv='score', method='levene')
            equal_var = bool(res['equal_var'].iloc[0])
            print("\tLevene's test (equal variances?):", equal_var)

            test_metrics[metric_name]['equal_var'] = equal_var

            if equal_var:
                # One-way ANOVA (equal variances)
                p = pg.anova(data=df_metric, between='config', dv='score').iloc[0]['p-unc']
                print(f"\tOne-way ANOVA, p<{p_sig}:", p < p_sig)
                test_metrics[metric_name]['anova'] = p

                if p < p_sig:
                    # Tukey HSD post-hoc (equal variances)
                    df_t = pg.pairwise_tukey(data=df_metric, between='config', dv='score')
                    test_metrics[metric_name]['tukey_hsd'] = df_t
                    test_metrics[metric_name]['winner'] = strict_winner(df_t, df_metric['config'].unique().tolist(), alpha=p_sig)
                    print(f"\tTukey HSD post-hoc, winner: {test_metrics[metric_name]['winner']}")

            else:
                # Welch’s ANOVA (unequal variances)
                p = pg.welch_anova(data=df_metric, between='config', dv='score').iloc[0]['p-unc']
                print(f"\tWelch's ANOVA, p<{p_sig}:", p < p_sig)
                test_metrics[metric_name]['welch_anova'] = p

                if p < p_sig:
                    # Games–Howell post-hoc (unequal variances)
                    df_gh = pg.pairwise_gameshowell(data=df_metric, between='config', dv='score')
                    test_metrics[metric_name]['games_howell'] = df_gh
                    test_metrics[metric_name]['winner'] = strict_winner(df_gh, df_metric['config'].unique().tolist(), alpha=p_sig)
                    print(f"\tGames–Howell post-hoc, winner: {test_metrics[metric_name]['winner']}")

        test_metrics[metric_name]['accumulated'] = df_metric.groupby('config')['score'].mean().to_dict()
        test_metrics[metric_name]['count'] = df_metric.groupby('config')['score'].count().to_dict()


ratescore, ratescore:
No. samples: 1624.
	Levene's test (equal variances?): False
	Welch's ANOVA, p<0.05: True
	Games–Howell post-hoc, winner: None
ratescore_chexpert_plus, ratescore:
No. samples: 62.
	Levene's test (equal variances?): True
	One-way ANOVA, p<0.05: True
	Tukey HSD post-hoc, winner: None


/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/emnli trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/cxrmate trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/cxrmate_rrg24 trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/cxrmate_ed trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/emnli trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/cxrmate trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiment

ratescore_rexgradient, ratescore:
No. samples: 10000.
	Levene's test (equal variances?): False
	Welch's ANOVA, p<0.05: True
	Games–Howell post-hoc, winner: None
green, green:
No. samples: 1624.
	Levene's test (equal variances?): False
	Welch's ANOVA, p<0.05: True
	Games–Howell post-hoc, winner: None
green_chexpert_plus, green:
No. samples: 62.
	Levene's test (equal variances?): True
	One-way ANOVA, p<0.05: True
	Tukey HSD post-hoc, winner: None


/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/emnli trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/cxrmate trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/cxrmate_rrg24 trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/cxrmate_ed trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/emnli trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/cxrmate trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiment

green_rexgradient, green:
No. samples: 10000.
	Levene's test (equal variances?): False
	Welch's ANOVA, p<0.05: True
	Games–Howell post-hoc, winner: None
radeval_bertscore, f1:
No. samples: 1624.
	Levene's test (equal variances?): False
	Welch's ANOVA, p<0.05: True
	Games–Howell post-hoc, winner: None
radeval_bertscore_chexpert_plus, f1:
No. samples: 62.
	Levene's test (equal variances?): False
	Welch's ANOVA, p<0.05: True
	Games–Howell post-hoc, winner: None


...

bertscore_rexgradient, f1:
No. samples: 10000.
	Levene's test (equal variances?): False
	Welch's ANOVA, p<0.05: True
	Games–Howell post-hoc, winner: None
bleu, bleu_4:
No. samples: 1624.
	Levene's test (equal variances?): False
	Welch's ANOVA, p<0.05: True
	Games–Howell post-hoc, winner: None
bleu_chexpert_plus, bleu_4:
No. samples: 62.
	Levene's test (equal variances?): False
	Welch's ANOVA, p<0.05: True
	Games–Howell post-hoc, winner: None


/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/emnli trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/cxrmate trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/cxrmate_rrg24 trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/cxrmate_ed trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/emnli trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/cxrmate trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiment

bleu_rexgradient, bleu_4:
No. samples: 10000.
	Levene's test (equal variances?): False
	Welch's ANOVA, p<0.05: True
	Games–Howell post-hoc, winner: None
rouge_l, f1:
No. samples: 1624.
	Levene's test (equal variances?): False
	Welch's ANOVA, p<0.05: True
	Games–Howell post-hoc, winner: None
rouge_l_chexpert_plus, f1:
No. samples: 62.
	Levene's test (equal variances?): False
	Welch's ANOVA, p<0.05: True
	Games–Howell post-hoc, winner: None


/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/emnli trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/cxrmate trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/cxrmate_rrg24 trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/cxrmate_ed trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/emnli trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiments/cxrmate2/final/cxrmate trial 0.
/tmp/ipykernel_48412/808153648.py:141: UserWarning: No CSV found for best epoch -1 in /scratch3/nic261/experiment

rouge_l_rexgradient, f1:
No. samples: 10000.
	Levene's test (equal variances?): False
	Welch's ANOVA, p<0.05: True
	Games–Howell post-hoc, winner: None


In [107]:
num_samples_table = pd.DataFrame({metric: test_metrics[metric]['count'] for metric in test_metrics.keys() if 'count' in test_metrics[metric]})
num_samples_table

,RaTEScore MIMIC-CXR,RaTEScore CheXpert Plus,RaTEScore ReXgradient,GREEN MIMIC-CXR,GREEN CheXpert Plus,GREEN ReXgradient,RadEval BERTScore MIMIC-CXR,RadEval BERTScore CheXpert Plus,RadEval BERTScore ReXgradient,SRR MIMIC-CXR,...,CXR-BERT ReXgradient,BERTScore MIMIC-CXR,BERTScore CheXpert Plus,BERTScore ReXgradient,BLEU MIMIC-CXR,BLEU CheXpert Plus,BLEU ReXgradient,ROUGE-L MIMIC-CXR,ROUGE-L CheXpert Plus,ROUGE-L ReXgradient
CXRMate,1624,NaN,NaN,1624,NaN,NaN,1624,NaN,NaN,1624.0,...,NaN,1624,NaN,NaN,1624,NaN,NaN,1624,NaN,NaN
CXRMate-2,1624,62.0,10000.0,1624,62.0,10000.0,1624,62.0,10000.0,1624.0,...,10000.0,1624,62.0,10000.0,1624,62.0,10000.0,1624,62.0,10000.0
CXRMate-2 (leg. A),1624,62.0,10000.0,1624,62.0,10000.0,1624,62.0,10000.0,1624.0,...,10000.0,1624,62.0,10000.0,1624,62.0,10000.0,1624,62.0,10000.0
CXRMate-ED,1624,NaN,NaN,1624,NaN,NaN,1624,NaN,NaN,1624.0,...,NaN,1624,NaN,NaN,1624,NaN,NaN,1624,NaN,NaN
CXRMate-RRG24,1624,NaN,NaN,1624,NaN,NaN,1624,NaN,NaN,1624.0,...,NaN,1624,NaN,NaN,1624,NaN,NaN,1624,NaN,NaN
EMNLI,1624,NaN,NaN,1624,NaN,NaN,1624,NaN,NaN,1624.0,...,NaN,1624,NaN,NaN,1624,NaN,NaN,1624,NaN,NaN
Libra,1624,62.0,10000.0,1624,62.0,10000.0,1624,62.0,10000.0,1624.0,...,10000.0,1624,62.0,10000.0,1624,62.0,10000.0,1624,62.0,10000.0
MAIRA-2,1624,62.0,NaN,1624,62.0,NaN,1624,62.0,NaN,1624.0,...,NaN,1624,62.0,NaN,1624,62.0,NaN,1624,62.0,NaN
MedGemma,1624,62.0,10000.0,1624,62.0,10000.0,1624,62.0,10000.0,1624.0,...,10000.0,1624,62.0,10000.0,1624,62.0,10000.0,1624,62.0,10000.0
MedVersa,1624,62.0,10000.0,1624,62.0,10000.0,1624,62.0,10000.0,1624.0,...,10000.0,1624,62.0,10000.0,1624,62.0,10000.0,1624,62.0,10000.0


In [108]:
metric_to_abbreviation = {
    'RaTEScore': 'RS',
    'GREEN': 'G',
    'CXR-BERT': 'CB',
    'RadGraph-XL': 'RG-XL',
    'CheXbert': 'CX',
    'SRR': 'SRR',
    'BERTScore': 'BS',
    'BLEU': 'B4',
    'ROUGE-L': 'R-L',
    'ARN': 'ARN',
    'RadEval BERTScore': 'RE-BS',
}
abbreviation_to_metric = {v: k for k, v in metric_to_abbreviation.items()}

In [109]:
table = pd.DataFrame({metric_name: test_metrics[metric_name]['accumulated'] for metric_name in test_metrics.keys() if 'accumulated' in test_metrics[metric_name]})
table = table.reindex([i['name'] for i in configs if i['name'] in table.index])
# table.insert(5, 'GREEN ReXgradient', float('nan'))
table.insert(
    0, 
    '\\#Train',
    [
        f"{i['training_examples']:,}" if isinstance(i['training_examples'], int) else '-'
        for i in configs if i['name'] in table.index
    ]
)
table.insert(
    0, 
    'Venue',
    [
        f"{i['venue']}" if 'venue' in i else '-'
        for i in configs if i['name'] in table.index
    ]
)
table

table.at['MAIRA-2', 'SRR ReXgradient'] = float('nan') 
table.at['MAIRA-2', 'CheXbert ReXgradient'] = float('nan') 
table.at['CXRMate-RRG24', 'SRR CheXpert Plus'] = float('nan') 
table.at['CXRMate-RRG24', 'CheXbert CheXpert Plus'] = float('nan') 
table

KeyError: 'training_examples'

In [ ]:
table[table.select_dtypes(include='float').columns] = table.select_dtypes(include='float').round(3) * 100
max_dict = table.select_dtypes(include='float').max().to_dict()
min_dict = table.select_dtypes(include='float').min().to_dict()

metric_names = list(dict.fromkeys(
    i.replace(' MIMIC-CXR', '').replace(' CheXpert Plus', '').replace(' ReXgradient', '')
    for i in table.columns
))

metric_names = [f'{metric_to_abbreviation[i]}' if i in metric_to_abbreviation else i for i in metric_names]

num_cols = len(metric_names) + 1

significant_model = 'CXRMate-2'

mimic_cxr = []
chexpert_plus = []
rexgradient = []

print(
    f'\\caption{{\\label{{tab:objective}}\\underline{{Underlined}} scores indicate a significant difference to the scores of `Images`. Evaluation is performed on the \\textbf{{findings}} section. {', '.join([f'{i}={abbreviation_to_metric[i]}' for i in metric_names if i not in ['\\#Train', 'Venue']])}.}}'
)
print(f'\\begin{{tabular}}{{l{"c" * (num_cols - 1)}}}')
print('\\toprule')

print('Model', end='')
[print(f' & {i}', end='') for i in metric_names]
print('\\\\')

for index, row in table.iterrows():
    
    has_mimic_cxr = not all([math.isnan(row[col]) for col in [col for col in row.index if 'MIMIC-CXR' in col]])
    has_chexpert_plus = not all([math.isnan(row[col]) for col in [col for col in row.index if 'CheXpert Plus' in col]])
    has_rexgradient = not all([math.isnan(row[col]) for col in [col for col in row.index if 'ReXgradient' in col]])

    if has_mimic_cxr:
        mimic_cxr.append(f'{index}')
    if has_chexpert_plus:
        chexpert_plus.append(f'{index}')
    if has_rexgradient:     
        rexgradient.append(f'{index}')
    toggle = True
    power_factor = 2
    for col_idx, i in enumerate(row):

        toggle = not toggle

        if 'MIMIC-CXR' in row.index[col_idx] and not has_mimic_cxr:
            continue
        if 'CheXpert Plus' in row.index[col_idx] and not has_chexpert_plus:
            continue
        if 'ReXgradient' in row.index[col_idx] and not has_rexgradient:
            continue
        
        metric = table.columns.to_list()[col_idx]
        if metric in max_dict:
            if math.isnan(i):
                i = '-'
                prefix = '&\\cellcolor[RGB]{255,255,255}'
            else:
                normalised = (i - min_dict[metric]) / (max_dict[metric] - min_dict[metric] + 1e-6)
                normalised = normalised ** power_factor
                rgb = seaborn.color_palette("light:#FFC20A".lower() if toggle else "light:#0C7BDC".lower(), as_cmap=True)(normalised, bytes=True)
                prefix = f'&\\cellcolor[RGB]{{{rgb[0]},{rgb[1]},{rgb[2]}}}'
        else:
            prefix = '&'

        value = f'{i:.1f}' if not isinstance(i, str) else f'{i}'

        if index == significant_model and metric in test_metrics.keys():
            if 'winner' in test_metrics[metric]:
                if test_metrics[metric]['winner'] == index:
                    value = f'\\underline{{{value}}}'

        if metric in max_dict and i == max_dict[metric]:
            value = f'\\textbf{{{value}}}'

        if 'MIMIC-CXR' in row.index[col_idx]:
            mimic_cxr[-1] += (f'{prefix}{value}')
        elif 'CheXpert Plus' in row.index[col_idx]:
            chexpert_plus[-1] += (f'{prefix}{value}')
        elif 'ReXgradient' in row.index[col_idx]:
            rexgradient[-1] += (f'{prefix}{value}')
        else:
            if has_mimic_cxr:
                mimic_cxr[-1] += (f'{prefix}{value}')
            if has_chexpert_plus:
                chexpert_plus[-1] += (f'{prefix}{value}')
            if has_rexgradient:
                rexgradient[-1] += (f'{prefix}{value}')

    if has_mimic_cxr:
        mimic_cxr[-1] += ('\\\\\n')
    if has_chexpert_plus:
        chexpert_plus[-1] += ('\\\\\n')
    if has_rexgradient:
        rexgradient[-1] += ('\\\\\n')

print(f'\\multicolumn{{{num_cols}}}{{c}}{{\\cellcolor[RGB]{{200,200,200}} \\textit{{\\textbf{{MIMIC-CXR}} $n=1\\,624$}}}} \\\\')
print(''.join(mimic_cxr))
print(f'\\multicolumn{{{num_cols}}}{{c}}{{\\cellcolor[RGB]{{200,200,200}} \\textit{{\\textbf{{CheXpert Plus}} $n=62$}}}} \\\\')
print(''.join(chexpert_plus))
print(f'\\multicolumn{{{num_cols}}}{{c}}{{\\cellcolor[RGB]{{200,200,200}} \\textit{{\\textbf{{ReXgradient}} $n=10\\,000$}}}} \\\\')
print(''.join(rexgradient))

\caption{\label{tab:objective}\underline{Underlined} scores indicate a significant difference to the scores of `Images`. Evaluation is performed on the \textbf{findings} section. RS=RaTEScore, G=GREEN, RE-BS=RadEval BERTScore, SRR=SRR, RG-XL=RadGraph-XL, CX=CheXbert, CB=CXR-BERT, BS=BERTScore, B4=BLEU, R-L=ROUGE-L.}
\begin{tabular}{lcccccccccccc}
\toprule
Model & Venue & \#Train & RS & G & RE-BS & SRR & RG-XL & CX & CB & BS & B4 & R-L\\
\multicolumn{13}{c}{\cellcolor[RGB]{200,200,200} \textit{\textbf{MIMIC-CXR} $n=1\,624$}} \\
EMNLI&NAACL`21&152,173&\cellcolor[RGB]{204,222,239}59.2&\cellcolor[RGB]{246,227,172}39.1&\cellcolor[RGB]{233,237,242}28.6&\cellcolor[RGB]{242,240,237}14.3&\cellcolor[RGB]{189,214,237}28.5&\cellcolor[RGB]{242,240,237}31.1&\cellcolor[RGB]{218,229,240}68.1&\cellcolor[RGB]{243,236,218}23.1&\cellcolor[RGB]{237,239,242}6.4&\cellcolor[RGB]{243,236,216}28.0\\
CXRMate&IMU`24&125,395&\cellcolor[RGB]{229,235,241}56.7&\cellcolor[RGB]{244,233,205}36.3&\cellcolor[RGB]{230,235,

In [ ]:
table

,Venue,\#Train,RaTEScore MIMIC-CXR,RaTEScore CheXpert Plus,RaTEScore ReXgradient,GREEN MIMIC-CXR,GREEN CheXpert Plus,GREEN ReXgradient,RadEval BERTScore MIMIC-CXR,RadEval BERTScore CheXpert Plus,...,CXR-BERT ReXgradient,BERTScore MIMIC-CXR,BERTScore CheXpert Plus,BERTScore ReXgradient,BLEU MIMIC-CXR,BLEU CheXpert Plus,BLEU ReXgradient,ROUGE-L MIMIC-CXR,ROUGE-L CheXpert Plus,ROUGE-L ReXgradient
EMNLI,NAACL`21,"152,173",59.2,NaN,NaN,39.1,NaN,NaN,28.6,NaN,...,NaN,23.1,NaN,NaN,6.4,NaN,NaN,28.0,NaN,NaN
CXRMate,IMU`24,"125,395",56.7,NaN,NaN,36.3,NaN,NaN,29.1,NaN,...,NaN,27.7,NaN,NaN,9.2,NaN,NaN,27.0,NaN,NaN
CXRMate-RRG24,BioNLP`24,"550,395",58.1,NaN,NaN,35.8,NaN,NaN,27.2,NaN,...,NaN,25.8,NaN,NaN,7.6,NaN,NaN,26.0,NaN,NaN
MAIRA-2,`24,"501,825",59.2,51.1,NaN,38.6,27.2,NaN,37.2,25.2,...,NaN,13.9,19.9,NaN,17.1,4.8,NaN,34.5,22.6,NaN
MedVersa,`24,-,60.6,49.4,55.9,38.8,23.2,43.4,33.3,20.4,...,30.6,26.2,1.6,25.4,16.0,3.5,4.4,34.6,20.1,24.4
Libra,ACL`25,"1,213,097",53.7,49.6,56.2,29.8,22.4,44.8,24.9,24.8,...,31.9,20.5,15.5,23.8,4.0,2.5,4.6,21.8,18.4,21.9
CXRMate-ED,ACL`25,"76,398",59.9,NaN,NaN,37.5,NaN,NaN,30.7,NaN,...,NaN,34.8,NaN,NaN,10.7,NaN,NaN,29.8,NaN,NaN
MedGemma,`25,"231,483",57.3,51.4,61.2,33.4,23.9,57.4,25.9,22.8,...,61.7,24.8,17.7,30.0,6.1,3.9,5.2,23.8,19.7,24.4
CXRMate-2 (leg. A),-,"313,503",67.5,59.2,68.8,47.3,34.3,57.8,45.9,38.3,...,78.6,45.3,32.6,47.9,24.5,13.8,28.4,42.2,31.1,45.4
CXRMate-2,-,"313,503",66.9,58.7,68.6,46.3,34.7,57.7,45.3,34.7,...,78.2,45.0,20.1,47.6,24.2,13.2,28.2,41.9,31.0,45.1
